# 06 — MEM Electricity | Bronze Ingestion (incremental, Auto Loader)

Capture every Ministry of Energy and Mines (MEM) **Informe de Desempeño — Anexos** workbook that arrives in the raw archive, exactly once.

| Item | Value |
|---|---|
| Archive | `/Volumes/rd_economic_intelligence/bronze/mem_raw/ede_performance/<yyyy>/<mm>/*.xlsx` (append-only, written by `06_MEM_Electricity_Download`) |
| Step 1 — discover | **Auto Loader** (`cloudFiles`, format `binaryFile`, `trigger(availableNow=True)`, shared helper `src/shared/ingestion.py`). The checkpoint remembers which files were already seen, so each run registers only new files in `bronze.mem_annex_files` (path, size, modification time, SHA-256) and stops. |
| Step 2 — capture | Every registered file not yet captured is read with openpyxl: one Bronze row per sheet row, all cells kept as a JSON array (dates as ISO text). Layout is interpreted in Silver. |
| Sheets captured | `EDE's` (monthly distributor indicators since 2009) and `Nuevo Regimen tarifario` (quarterly tariffs since 2020) |
| History | One snapshot per archived file, identified by `_source_path`. Older annexes stay in Bronze for auditing revisions; Silver uses the newest. |
| Recovery | Step 2 compares the file register with what Bronze already holds, so a file registered by a run that failed before capture is captured on the next run. Each file is written in one append. |
| Target | `rd_economic_intelligence.bronze.mem_electricity_raw_rows` |

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import datetime, date, timezone
import sys, uuid, json, importlib
import openpyxl
from pyspark.sql import functions as F

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import ingestion as ING
importlib.reload(ING)

CATALOG = "rd_economic_intelligence"
RAW_PATH = f"/Volumes/{CATALOG}/bronze/mem_raw/ede_performance/"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/mem_raw/_checkpoints/ede_performance_files/"
FILES = f"{CATALOG}.bronze.mem_annex_files"
TARGET = f"{CATALOG}.bronze.mem_electricity_raw_rows"
SHEETS = ["EDE's", "Nuevo Regimen tarifario"]
BATCH_ID, INGESTION_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

## Step 1 — Auto Loader: register new files · Step 2 — find registered files not yet in Bronze

In [0]:
registered = ING.register_new_files(spark, RAW_PATH, "*.xlsx", CHECKPOINT, FILES,
    comment="Bronze: register of MEM annex workbooks found in the raw archive by Auto Loader (one row per file).")
pending = ING.pending_files(spark, FILES, TARGET)
print(f"Auto Loader registered {registered} new file(s) · pending capture: {[f['_source_file'] for f in pending]}")

## Capture each pending file

In [0]:
def cell(v):
    if isinstance(v, (datetime, date)):
        return v.isoformat()
    return None if v is None else (v if isinstance(v, (int, float)) else str(v))

captured = {}
for f in pending:
    wb = openpyxl.load_workbook(ING.local_path(f["_source_path"]), data_only=True)
    missing = [s for s in SHEETS if s not in wb.sheetnames]
    if missing:
        raise ValueError(f"{f['_source_file']}: structural contract failed, sheets missing: {missing}")
    rows = []
    for s in SHEETS:
        for i, r in enumerate(wb[s].iter_rows(values_only=True), start=1):
            vals = [cell(v) for v in r]
            while vals and vals[-1] is None:
                vals.pop()
            if any(v is not None for v in vals):
                rows.append((s, i, json.dumps(vals, ensure_ascii=False)))
    df = ING.lineage(spark.createDataFrame(rows, "_source_sheet string, _source_row_number int, row_json string"), f, BATCH_ID, INGESTION_TS)
    df.write.format("delta").mode("append").saveAsTable(TARGET)  # one atomic append per file
    captured[f["_source_file"]] = len(rows)

if spark.catalog.tableExists(TARGET):
    spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Bronze: MEM Informe de Desempeño annex rows (EDE monthly indicators and tariff charges), one snapshot per archived file, as published.'")
snapshots = spark.table(TARGET).select("_source_path").distinct().count() if spark.catalog.tableExists(TARGET) else 0
dbutils.notebook.exit(json.dumps({"status": "SUCCESS" if captured else "NO_NEW_FILES", "registered_by_auto_loader": registered,
                                  "captured_rows": captured, "snapshots_in_bronze": snapshots}, ensure_ascii=False))